In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Load JSON
# -----------------------------
json_path = r"C:\Users\ae\OneDrive\thesis\data\merged.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# -----------------------------
# Calculate sentence lengths
# -----------------------------
df = pd.DataFrame(data)

# Number of words
df["text_word_length"] = df["text"].apply(lambda x: len(str(x).split()))
df["standardized_word_length"] = df["standardized"].apply(lambda x: len(str(x).split()))

# Number of characters
df["text_char_length"] = df["text"].apply(len)
df["standardized_char_length"] = df["standardized"].apply(len)

# -----------------------------
# Summary Statistics
# -----------------------------
print("===== ORIGINAL TEXT (Words) =====")
print(df["text_word_length"].describe())

print("\n===== STANDARDIZED TEXT (Words) =====")
print(df["standardized_word_length"].describe())

print("\n===== ORIGINAL TEXT (Characters) =====")
print(df["text_char_length"].describe())

print("\n===== STANDARDIZED TEXT (Characters) =====")
print(df["standardized_char_length"].describe())

# -----------------------------
# Percentiles
# -----------------------------
percentiles = [0.5, 0.9, 0.95, 0.99]

print("\nOriginal Text Word Length Percentiles")
print(df["text_word_length"].quantile(percentiles))

print("\nStandardized Text Word Length Percentiles")
print(df["standardized_word_length"].quantile(percentiles))

# -----------------------------
# Maximum Length Sentences
# -----------------------------
print("\nLongest Original Sentence:")
idx = df["text_word_length"].idxmax()
print(df.loc[idx, ["id", "text", "text_word_length"]])

print("\nLongest Standardized Sentence:")
idx = df["standardized_word_length"].idxmax()
print(df.loc[idx, ["id", "standardized", "standardized_word_length"]])

# -----------------------------
# Histogram
# -----------------------------
plt.figure(figsize=(10,5))
plt.hist(df["text_word_length"], bins=40)
plt.title("Distribution of Original Sentence Lengths")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

plt.figure(figsize=(10,5))
plt.hist(df["standardized_word_length"], bins=40)
plt.title("Distribution of Standardized Sentence Lengths")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

In [ ]:
df["text_word_length"] = df["text"].apply(lambda x: len(str(x).split()))

In [ ]:
print(df["text_word_length"].describe())

punctuation distribution

In [2]:
import json
import pandas as pd
import re

import json
import pandas as pd
import re

with open(r"C:\Users\ae\OneDrive\thesis\data\mc_joint_text_restoration.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

# Regex for punctuation
punctuation_pattern = re.compile(r"[.,!?;:'\"()\[\]{}\-…]")

# Check if sentence contains punctuation
df["has_punctuation"] = df["text"].apply(
    lambda x: bool(punctuation_pattern.search(str(x)))
)

# Counts
no_punctuation = (~df["has_punctuation"]).sum()
with_punctuation = df["has_punctuation"].sum()

print(f"Total sentences      : {len(df)}")
print(f"With punctuation     : {with_punctuation}")
print(f"Without punctuation  : {no_punctuation}")

# Percentages
print(f"\nWith punctuation     : {with_punctuation/len(df)*100:.2f}%")
print(f"Without punctuation  : {no_punctuation/len(df)*100:.2f}%")

Total sentences      : 13830
With punctuation     : 6872
Without punctuation  : 6958

With punctuation     : 49.69%
Without punctuation  : 50.31%


## Step 1 - Data exclusion

Count the amount of data excluded for each scenario

Check original punctuated vs unpunctuated

In [1]:
import json
import re

file_path = r"C:\Users\ae\OneDrive\thesis\data\raw_fb_data.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

total_sentences = 0
total_comments = len(data)

sentence_counts = []

for row in data:
    text = row.get("text", "").strip()

    if not text:
        sentence_counts.append(0)
        continue

    # Split sentences based on ., !, ?, and ellipses
    sentences = re.split(r'(?<=[.!?])\s+', text)

    # Remove empty items
    sentences = [s.strip() for s in sentences if s.strip()]

    sentence_count = len(sentences)

    sentence_counts.append(sentence_count)
    total_sentences += sentence_count

print(f"Total comments: {total_comments}")
print(f"Total sentences: {total_sentences}")
print(f"Average sentences per comment: {total_sentences / total_comments:.2f}")

Total comments: 15438
Total sentences: 21943
Average sentences per comment: 1.42


This code gives rows containing emojis + text

In [16]:
import json
import re

# Input JSON file
input_file = r"C:\Users\ae\OneDrive\thesis\data\raw_fb_data.json"

# Output text file
output_file = r"C:\Users\ae\OneDrive\thesis\data\emoji_and_nan_rows.txt"


# Emoji detection pattern
emoji_pattern = re.compile(
    "["
    "\U0001F300-\U0001F5FF"  # Symbols & pictographs
    "\U0001F600-\U0001F64F"  # Emoticons
    "\U0001F680-\U0001F6FF"  # Transport & map symbols
    "\U0001F700-\U0001F77F"  # Alchemical symbols
    "\U0001F780-\U0001F7FF"  # Geometric shapes
    "\U0001F800-\U0001F8FF"  # Supplemental arrows
    "\U0001F900-\U0001F9FF"  # Supplemental symbols & pictographs
    "\U0001FA00-\U0001FAFF"  # Extended pictographs
    "\U00002600-\U000026FF"  # Miscellaneous symbols
    "\U00002700-\U000027BF"  # Dingbats
    "\U00002300-\U000023FF"  # Miscellaneous technical symbols
    "\U00002B00-\U00002BFF"  # Arrows and symbols
    "\U0001F1E6-\U0001F1FF"  # Regional indicator symbols / flags
    "]"
)


def contains_emoji(text):
    """
    Detects whether a string contains at least one emoji.
    """
    return bool(emoji_pattern.search(text))


# Load JSON
with open(input_file, "r", encoding="utf-8") as file:
    data = json.load(file)


emoji_rows = []
nan_rows = []


for row in data:

    text = str(row.get("text", ""))

    # Find rows containing any emoji
    if contains_emoji(text):
        emoji_rows.append(row)

    # Find rows where text is exactly 'nan'
    if text.strip().lower() == "nan":
        nan_rows.append(row)


# Write results to text file
with open(output_file, "w", encoding="utf-8") as file:

    file.write("=" * 70 + "\n")
    file.write("ROWS CONTAINING EMOJIS\n")
    file.write("=" * 70 + "\n\n")

    for row in emoji_rows:
        file.write(json.dumps(row, ensure_ascii=False, indent=4))
        file.write(",\n\n")


    file.write("\n\n")
    file.write("=" * 70 + "\n")
    file.write("ROWS WHERE TEXT IS 'nan'\n")
    file.write("=" * 70 + "\n\n")

    for row in nan_rows:
        file.write(json.dumps(row, ensure_ascii=False, indent=4))
        file.write(",\n\n")


print("Finished!")
print(f"Rows containing emojis: {len(emoji_rows)}")
print(f"Rows where text is 'nan': {len(nan_rows)}")
print(f"Results saved to: {output_file}")

Finished!
Rows containing emojis: 3267
Rows where text is 'nan': 375
Results saved to: C:\Users\ae\OneDrive\thesis\data\emoji_and_nan_rows.txt


Lines containing emojis only

In [20]:
import json
import re

# Input JSON file
input_file = r"C:\Users\ae\OneDrive\thesis\data\raw_fb_data.json"

# Output text file
output_file = r"C:\Users\ae\OneDrive\thesis\data\emoji_only_and_nan_rows.txt"


# Unicode emoji pattern
emoji_pattern = re.compile(
    r"^[\s"
    r"\U0001F300-\U0001F5FF"
    r"\U0001F600-\U0001F64F"
    r"\U0001F680-\U0001F6FF"
    r"\U0001F700-\U0001F77F"
    r"\U0001F780-\U0001F7FF"
    r"\U0001F800-\U0001F8FF"
    r"\U0001F900-\U0001F9FF"
    r"\U0001FA00-\U0001FAFF"
    r"\u2600-\u26FF"
    r"\u2700-\u27BF"
    r"\u2300-\u23FF"
    r"\u2B00-\u2BFF"
    r"\U0001F1E6-\U0001F1FF"
    r"\u200D"
    r"\uFE0F"
    r"\U0001F3FB-\U0001F3FF"
    r"]+$"
)


def is_emoji_only(text):
    if not text or not text.strip():
        return False

    return bool(emoji_pattern.fullmatch(text))


# Load JSON
with open(input_file, "r", encoding="utf-8") as file:
    data = json.load(file)


emoji_only_rows = []
nan_rows = []


for row in data:

    text = str(row.get("text", ""))

    # Emoji-only rows
    if is_emoji_only(text):
        emoji_only_rows.append(row)

    # Rows where text is exactly "nan"
    if text.strip().lower() == "nan":
        nan_rows.append(row)


# Function to write rows with commas between objects
def write_json_objects(file, rows):

    for i, row in enumerate(rows):

        file.write(json.dumps(row, ensure_ascii=False, indent=4))

        # Add comma between objects
        if i < len(rows) - 1:
            file.write(",")

        # Only one newline
        file.write("\n")


# Write output
with open(output_file, "w", encoding="utf-8") as file:

    file.write("=" * 70 + "\n")
    file.write("EMOJI-ONLY ROWS\n")
    file.write("=" * 70 + "\n\n")

    write_json_objects(file, emoji_only_rows)


    file.write("\n")
    file.write("=" * 70 + "\n")
    file.write("ROWS WHERE TEXT IS 'nan'\n")
    file.write("=" * 70 + "\n\n")

    write_json_objects(file, nan_rows)


print("Finished!")
print(f"Emoji-only rows: {len(emoji_only_rows)}")
print(f"'nan' rows: {len(nan_rows)}")
print(f"Results saved to: {output_file}")

Finished!
Emoji-only rows: 165
'nan' rows: 375
Results saved to: C:\Users\ae\OneDrive\thesis\data\emoji_only_and_nan_rows.txt


DATA EXCLUSION

In [24]:
import pandas as pd

# Path to your text file
file_path = r"C:\Users\ae\OneDrive\thesis\data\data_exclusion.txt"

# Categories exactly as they appear in your file
categories = [    
    "ENGLISH",                          # Fully English comments 
    "FRENCH",                           # Fully French comments  
    "OTHER LANGUAGES",                  # Comments in other languages
    "NAME TAGGING",                     # Name-only comments
    "EMOJIS" ,                          # Emoji-only comments   
    "LINKS",                            # Link-only comments
    "TECHNICAL ARTEFACTS",              # NAN entries
    "NEAR-DUPLICATES",                  # Near-duplicate comments
    "UNINTELLIGIBLE COMMENTS",          # Comments whose intended meaning could not be reliably determined by native speakers
    "HARSH",                            # Highly vulgar comments
    "CODE-SWITCHED ENGLISH/FRENCH",     # English/French sentences
]

# Create dictionary with count = 0
counts = {category: 0 for category in categories}

current_category = None

with open(file_path, "r", encoding="utf-8") as file:

    for line in file:

        # Remove spaces and newline characters
        line = line.strip()

        # Detect category headers
        if line.startswith("//"):

            # Remove // and any spaces after it
            category = line[2:].strip().upper()

            # Check if it is one of our categories
            if category in counts:
                current_category = category

        # Count every JSON object under the current category
        elif current_category and line.startswith('"id":'):
            counts[current_category] += 1


# Create DataFrame
df = pd.DataFrame(
    list(counts.items()),
    columns=["Category", "Number of Sentences"]
)

# Print total
print("\nTOTAL:", df["Number of Sentences"].sum())
# Display the results
print(df.to_string(index=False))


TOTAL: 2255
                    Category  Number of Sentences
                     ENGLISH                  352
                      FRENCH                  944
             OTHER LANGUAGES                    6
                NAME TAGGING                   25
                      EMOJIS                  165
                       LINKS                    4
         TECHNICAL ARTEFACTS                  481
             NEAR-DUPLICATES                  204
     UNINTELLIGIBLE COMMENTS                   14
                       HARSH                   37
CODE-SWITCHED ENGLISH/FRENCH                   23


This code allows to check for the distribution of punctuation in the standardized text

Distribution of punctuation

In [7]:
import json
import unicodedata
from collections import Counter

def analyze_punctuation_distribution(json_file):
    """
    Analyze every punctuation mark that actually appears in the
    'standardized' field of the dataset.
    """

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    punct_counter = Counter()
    total_sentences = len(data)
    sentences_with_punct = 0
    total_punct = 0

    for item in data:
        sentence = item["standardized"]

        sentence_punct = []

        for ch in sentence:
            # Unicode categories starting with P are punctuation
            if unicodedata.category(ch).startswith("P"):
                punct_counter[ch] += 1
                sentence_punct.append(ch)

        if sentence_punct:
            sentences_with_punct += 1
            total_punct += len(sentence_punct)

    print("=" * 70)
    print("PUNCTUATION DISTRIBUTION (STANDARDIZED FIELD ONLY)")
    print("=" * 70)

    print(f"Total sentences              : {total_sentences}")
    print(f"Sentences with punctuation   : {sentences_with_punct}")
    print(f"Total punctuation marks      : {total_punct}")
    print(f"Average punctuation/sentence : {total_punct/total_sentences:.3f}")

    print("\nDistribution")
    print("-" * 70)
    print(f"{'Punctuation':<15}{'Unicode Name':<40}{'Count':>8}{'%':>8}")
    print("-" * 70)

    for p, c in punct_counter.most_common():
        name = unicodedata.name(p, "UNKNOWN")
        pct = c / total_punct * 100
        print(f"{repr(p):<15}{name:<40}{c:>8}{pct:>7.2f}")

    return punct_counter

In [8]:
analyze_punctuation_distribution(
    r"C:\Users\ae\OneDrive\thesis\data\mc_joint_text_restoration.json"
)

PUNCTUATION DISTRIBUTION (STANDARDIZED FIELD ONLY)
Total sentences              : 13830
Sentences with punctuation   : 13830
Total punctuation marks      : 53004
Average punctuation/sentence : 3.833

Distribution
----------------------------------------------------------------------
Punctuation    Unicode Name                               Count       %
----------------------------------------------------------------------
'.'            FULL STOP                                  29203  55.10
','            COMMA                                       9587  18.09
'?'            QUESTION MARK                               4398   8.30
'['            LEFT SQUARE BRACKET                         2510   4.74
']'            RIGHT SQUARE BRACKET                        2510   4.74
'!'            EXCLAMATION MARK                            1973   3.72
"'"            APOSTROPHE                                  1044   1.97
':'            COLON                                        586   1.11
'’'  

Counter({'.': 29203,
         ',': 9587,
         '?': 4398,
         '[': 2510,
         ']': 2510,
         '!': 1973,
         "'": 1044,
         ':': 586,
         '’': 223,
         '-': 215,
         '/': 176,
         '…': 109,
         ';': 93,
         '%': 74,
         ')': 70,
         '(': 69,
         '&': 31,
         '#': 30,
         '"': 24,
         '“': 15,
         '*': 14,
         '«': 11,
         '”': 11,
         '_': 10,
         '‘': 7,
         '»': 3,
         '@': 3,
         '—': 2,
         '‼': 2,
         '⁉': 1})

In [5]:
import json
import unicodedata

def extract_sentences_without_punctuation(input_file, output_file):
    """
    Extract all records whose 'standardized' field contains
    no Unicode punctuation characters.
    """

    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    no_punctuation = []

    for item in data:
        sentence = item["standardized"]

        has_punctuation = any(
            unicodedata.category(ch).startswith("P")
            for ch in sentence
        )

        if not has_punctuation:
            no_punctuation.append(item)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(no_punctuation, f, ensure_ascii=False, indent=4)

    print("=" * 60)
    print(f"Total sentences: {len(data)}")
    print(f"Sentences WITHOUT punctuation: {len(no_punctuation)}")
    print(f"Saved to: {output_file}")

In [6]:
extract_sentences_without_punctuation(
    r"C:\Users\ae\OneDrive\thesis\data\mc_joint_text_restoration.json",
    r"C:\Users\ae\OneDrive\thesis\data\sentences_without_punctuation.json"
)

Total sentences: 13830
Sentences WITHOUT punctuation: 0
Saved to: C:\Users\ae\OneDrive\thesis\data\sentences_without_punctuation.json
